# 通用因子回测模板

这个 Notebook 是因子生产与回测之间的明确分界：

- **因子 Notebook（第 1～5 步）**：只负责生成截至信号日可知的因子；
- **本模板（原第 6～9 步）**：构造未来收益、做 IC 检验、分组回测和绩效分析。

模板复用同目录 `backtest.py` 的严格回测引擎，支持两种因子输入：

1. 单个 Parquet：至少包含 `date`、`code` 和因子列；
2. 逐日 CSV 目录：文件名为 `YYYY-MM-DD.csv`，至少包含 `code` 和因子列。

当前配置为 UPSHADOW（训练期 2006–2017）。更换因子时通常只需修改下面的“因子配置区”。


## 1. 因子与回测配置

In [ ]:
from pathlib import Path
from IPython.display import display
import importlib.util
import json
import os
import sys

import numpy as np
import pandas as pd

LEARNING_DIR = Path(os.getenv('QUANT_LEARNING_DIR', str(next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd()))))
BACKTEST_DIR = LEARNING_DIR / 'Back_test'
BACKTEST_ENGINE = BACKTEST_DIR / 'backtest.py'

# 用明确文件路径加载本地引擎，避免导入环境中其他同名 backtest 包，
# 也避免重启内核后依赖旧会话里残留的模块。
assert BACKTEST_ENGINE.is_file(), f'回测引擎不存在：{BACKTEST_ENGINE}'
engine_spec = importlib.util.spec_from_file_location(
    'strict_factor_backtest_engine', BACKTEST_ENGINE
)
assert engine_spec is not None and engine_spec.loader is not None
backtest_engine = importlib.util.module_from_spec(engine_spec)
sys.modules[engine_spec.name] = backtest_engine
engine_spec.loader.exec_module(backtest_engine)

BacktestConfig = backtest_engine.BacktestConfig
DataPaths = backtest_engine.DataPaths
FactorSpec = backtest_engine.FactorSpec
StrictFactorBacktester = backtest_engine.StrictFactorBacktester


# ==================== 因子配置区：换因子主要改这里 ====================
FACTOR_NAME = 'volume_price_divergence'
FACTOR_COLUMN = 'volume_price_divergence_factor'
FACTOR_DIRECTION = 'positive'  # positive=因子越大越看多；negative=越大越看空
FACTOR_PREPROCESSED = True     # 已做 MAD/Z-score 则为 True；原始因子为 False

# 可填单个 Parquet 文件，也可填旧格式的逐日 CSV 文件夹。
default_factor_source = (
    LEARNING_DIR / 'Factor' / 'volume_price_divergence_output_20060104_20171229'
    / 'volume_price_divergence_factor_20060104_20171229.parquet'
)
FACTOR_SOURCE = Path(os.getenv('QUANT_FACTOR_SOURCE', str(default_factor_source)))
FACTOR_DIRECTION = FACTOR_DIRECTION.strip().lower()
# ====================================================================


# ========================= 回测配置区 ================================
BACKTEST_START_DATE = os.getenv('QUANT_BACKTEST_START_DATE', '2006-01-04')
BACKTEST_END_DATE = os.getenv('QUANT_BACKTEST_END_DATE', '2017-12-29')

HORIZON = 5             # VWAP 持有交易日数
EXECUTION_LAG = 1       # T 日收盘信号，至少 T+1 成交
REBALANCE_STEP = 5      # 必须 >= HORIZON，防止重叠收益复利
GROUPS = 10
COST_BPS = 10.0
MIN_STOCKS = 100

TRADABILITY_MODE = 'basic'
MISSING_RETURN_POLICY = 'zero'
MAX_MISSING_RETURN_RATIO = 0.40

# 仅当输入原始因子时，这些横截面处理参数才生效。
MAD_N = 5.0
STANDARDIZE = True
NEUTRALIZE = False
NEUTRALIZE_COLUMNS = ('log_market_cap',)

# False 会禁止使用 END_DATE 之后的行情补区间末端收益，隔离验证/测试集。
ALLOW_EXIT_AFTER_END = False
SAVE_HOLDINGS = False
# ====================================================================


default_market_parquet = LEARNING_DIR / 'Data' / 'stocks_daily_04-25.parquet'
MARKET_PARQUET = Path(os.getenv('QUANT_MARKET_PARQUET', str(default_market_parquet)))
CACHE_DIR = Path(os.getenv(
    'QUANT_BACKTEST_CACHE_DIR',
    str(BACKTEST_DIR / '.cache' / 'market_data'),
))
sample_tag = (
    f'{HORIZON}d_'
    f'{BACKTEST_START_DATE.replace("-", "")}_'
    f'{BACKTEST_END_DATE.replace("-", "")}'
)
default_output_dir = BACKTEST_DIR / 'outputs' / FACTOR_NAME / sample_tag
OUTPUT_DIR = Path(os.getenv('QUANT_BACKTEST_OUTPUT_DIR', str(default_output_dir)))

assert FACTOR_DIRECTION in {'positive', 'negative'}, (
    'FACTOR_DIRECTION 只能是 positive 或 negative'
)
assert pd.Timestamp(BACKTEST_START_DATE) <= pd.Timestamp(BACKTEST_END_DATE)
assert FACTOR_SOURCE.exists(), f'因子数据源不存在：{FACTOR_SOURCE}'
assert MARKET_PARQUET.exists(), f'行情文件不存在：{MARKET_PARQUET}'

print('回测引擎：', BACKTEST_ENGINE)
print('因子数据源：', FACTOR_SOURCE)
print('因子列/方向：', FACTOR_COLUMN, '/', FACTOR_DIRECTION)
print('行情文件：', MARKET_PARQUET)
print('回测区间：', BACKTEST_START_DATE, '至', BACKTEST_END_DATE)
print('结果目录：', OUTPUT_DIR)


## 2. 输入契约检查

这里仅检查结构和日期覆盖，不计算未来收益。单文件 Parquet 会由引擎只读取 `date/code/因子列`，不会载入因子文件中的其他宽表字段。


In [ ]:
import pyarrow.parquet as pq


def parquet_stat_range(parquet_file, column):
    column_index = parquet_file.schema_arrow.get_field_index(column)
    assert column_index >= 0, f'Parquet 缺少列：{column}'
    statistics = [
        parquet_file.metadata.row_group(i).column(column_index).statistics
        for i in range(parquet_file.metadata.num_row_groups)
    ]
    statistics = [item for item in statistics if item is not None and item.has_min_max]
    assert statistics, f'Parquet 的 {column} 列没有可用统计信息'
    return min(item.min for item in statistics), max(item.max for item in statistics)


requested_start = pd.Timestamp(BACKTEST_START_DATE)
requested_end = pd.Timestamp(BACKTEST_END_DATE)

if FACTOR_SOURCE.is_file():
    factor_file = pq.ParquetFile(FACTOR_SOURCE)
    factor_schema = factor_file.schema_arrow
    required_columns = {'date', 'code', FACTOR_COLUMN}
    missing_columns = required_columns - set(factor_schema.names)
    assert not missing_columns, f'因子 Parquet 缺少列：{sorted(missing_columns)}'

    factor_min_raw, factor_max_raw = parquet_stat_range(factor_file, 'date')
    factor_min = pd.Timestamp(factor_min_raw)
    factor_max = pd.Timestamp(factor_max_raw)
    assert factor_min <= requested_start, (
        f'因子起始日 {factor_min.date()} 晚于回测起始日 {requested_start.date()}'
    )
    assert factor_max >= requested_end, (
        f'因子结束日 {factor_max.date()} 早于回测结束日 {requested_end.date()}'
    )

    input_summary = pd.Series({
        '输入模式': 'single_parquet',
        'Parquet总行数': factor_file.metadata.num_rows,
        'Parquet列数': len(factor_schema.names),
        '本次读取列': f'date, code, {FACTOR_COLUMN}',
        '因子日期范围': f'{factor_min.date()} 至 {factor_max.date()}',
        '因子已预处理': FACTOR_PREPROCESSED,
    })
else:
    csv_files = sorted(FACTOR_SOURCE.glob('*.csv'))
    assert csv_files, '逐日因子目录中没有 CSV'
    csv_dates = pd.to_datetime([file.stem for file in csv_files], errors='coerce')
    assert not pd.isna(csv_dates).any(), '存在文件名不是 YYYY-MM-DD 的 CSV'
    assert csv_dates.min() <= requested_start and csv_dates.max() >= requested_end, (
        '逐日因子文件没有完整覆盖回测区间'
    )
    input_summary = pd.Series({
        '输入模式': 'daily_csv',
        'CSV文件数': len(csv_files),
        '首个文件': csv_files[0].name,
        '最后文件': csv_files[-1].name,
        '因子已预处理': FACTOR_PREPROCESSED,
    })

market_file = pq.ParquetFile(MARKET_PARQUET)
market_min_raw, market_max_raw = parquet_stat_range(market_file, '交易日')
market_min = pd.to_datetime(str(int(market_min_raw)), format='%Y%m%d')
market_max = pd.to_datetime(str(int(market_max_raw)), format='%Y%m%d')
assert market_min <= requested_start, (
    f'行情起始日 {market_min.date()} 晚于回测起始日 {requested_start.date()}'
)
assert market_max >= requested_end, (
    f'行情结束日 {market_max.date()} 早于回测结束日 {requested_end.date()}'
)

display(input_summary.to_frame('值'))
display(pd.Series({
    '行情日期范围': f'{market_min.date()} 至 {market_max.date()}',
    '请求回测范围': f'{requested_start.date()} 至 {requested_end.date()}',
}).to_frame('值'))


## 3. 运行严格通用回测

这一步对应原第 6～9 步。引擎按以下顺序执行：

```text
读取T日因子 → 横截面处理/分组 → 检查T+1可交易性
→ 读取T+1至T+1+HORIZON的复权VWAP收益 → IC/组合收益/成本/审计
```

首次运行会从 `stocks_daily.parquet` 建立年度轻量行情缓存，时间较长；以后不同因子共用同一缓存。

In [ ]:
paths = DataPaths(
    factor_dir=FACTOR_SOURCE,
    market_parquet=MARKET_PARQUET,
    market_cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR,
)

factor = FactorSpec(
    name=FACTOR_NAME,
    column=FACTOR_COLUMN,
    direction=FACTOR_DIRECTION,
    preprocessed=FACTOR_PREPROCESSED,
)

config = BacktestConfig(
    start_date=BACKTEST_START_DATE,
    end_date=BACKTEST_END_DATE,
    horizon=HORIZON,
    execution_lag=EXECUTION_LAG,
    rebalance_step=REBALANCE_STEP,
    groups=GROUPS,
    cost_bps=COST_BPS,
    min_stocks=MIN_STOCKS,
    tradability_mode=TRADABILITY_MODE,
    missing_return_policy=MISSING_RETURN_POLICY,
    max_missing_return_ratio=MAX_MISSING_RETURN_RATIO,
    mad_n=MAD_N,
    standardize=STANDARDIZE,
    neutralize=NEUTRALIZE,
    neutralize_columns=NEUTRALIZE_COLUMNS,
    market_exchanges=('SH', 'SZ'),
    save_holdings=SAVE_HOLDINGS,
    allow_exit_after_end=ALLOW_EXIT_AFTER_END,
)

result = StrictFactorBacktester(paths, factor, config).run(save=True)

print('回测完成')
print(
    f'实际信号区间：{result.group_returns.index.min():%Y-%m-%d} '
    f'至 {result.group_returns.index.max():%Y-%m-%d}'
)
print('有效回测日：', len(result.group_returns))
print('结果目录：', OUTPUT_DIR)

## 4. 完整性审计（先看这里）

先确认信号、成交和退出日期严格递增，再检查成交比例及缺失收益比例。审计不通过时不要先看收益曲线。

In [ ]:
audit = result.audit.copy()

assert (audit['entry_date'] > audit.index).all(), '存在同日或提前成交'
assert (audit['exit_date'] > audit['entry_date']).all(), '退出日期错误'
if not ALLOW_EXIT_AFTER_END:
    assert audit['exit_date'].max() <= pd.Timestamp(BACKTEST_END_DATE)

audit_summary = audit[[
    'factor_rows_raw', 'factor_rows_valid', 'fill_ratio', 'missing_return_ratio'
]].describe().T

display(audit_summary)
display(audit.head())

## 5. RankIC 与年度稳定性

In [ ]:
ic_series = result.ic_series.copy()
ic_summary = result.ic_summary.copy()

yearly_ic = ic_series.groupby(ic_series.index.year).agg(
    rank_ic_mean=('rank_ic', 'mean'),
    rank_ic_std=('rank_ic', 'std'),
    negative_ratio=('rank_ic', lambda x: (x < 0).mean()),
    observations=('rank_ic', 'count'),
    mean_stocks=('n_ic', 'mean'),
)
yearly_ic.index.name = 'year'
yearly_ic['rank_ic_ir'] = (
    yearly_ic['rank_ic_mean']
    / yearly_ic['rank_ic_std'].where(yearly_ic['rank_ic_std'] > 0)
    * np.sqrt(252 / REBALANCE_STEP)
)

display(ic_summary)
display(yearly_ic)

## 6. 分组收益、换手和交易成本

In [ ]:
group_returns = result.group_returns.copy()
group_columns = [f'Q{i}' for i in range(1, GROUPS + 1)]

group_summary = pd.DataFrame({
    'daily_mean': group_returns[group_columns].mean(),
    'annualized_mean': group_returns[group_columns].mean() * (252 / REBALANCE_STEP),
})
turnover_cost_summary = group_returns[[
    'long_turnover', 'short_turnover', 'transaction_cost',
    'long_short_gross', 'long_short_net'
]].describe().T

display(group_summary)
display(turnover_cost_summary)

## 7. 绩效与通用可视化

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / '.matplotlib'))
import matplotlib.pyplot as plt

performance = result.performance.copy()
display(performance)

if FACTOR_DIRECTION == 'negative':
    long_group, short_group = 'Q1', f'Q{GROUPS}'
else:
    long_group, short_group = f'Q{GROUPS}', 'Q1'

nav_columns = [long_group, short_group, 'long_short_net']
nav = (1 + group_returns[nav_columns].fillna(0)).cumprod()
rolling_rank_ic = ic_series['rank_ic'].rolling(60, min_periods=20).mean()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

rolling_rank_ic.plot(ax=axes[0, 0], linewidth=1, color='tab:blue')
axes[0, 0].axhline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('60-day Rolling RankIC')

group_summary['annualized_mean'].plot.bar(ax=axes[0, 1], color='tab:orange')
axes[0, 1].set_title('Annualized Mean Return by Group')
axes[0, 1].tick_params(axis='x', rotation=0)

nav.plot(ax=axes[1, 0], linewidth=1.2)
axes[1, 0].axhline(1, color='black', linewidth=0.8)
axes[1, 0].set_title('Net Asset Value')

audit[['fill_ratio', 'missing_return_ratio']].plot(ax=axes[1, 1], linewidth=1)
axes[1, 1].set_title('Execution Integrity')

for ax in axes.flat:
    ax.grid(alpha=0.25)
    ax.set_xlabel('')

plt.tight_layout()
dashboard_path = OUTPUT_DIR / 'research_dashboard.png'
fig.savefig(dashboard_path, dpi=160, bbox_inches='tight')
plt.show()

yearly_ic.to_csv(OUTPUT_DIR / 'yearly_ic.csv')
group_summary.to_csv(OUTPUT_DIR / 'group_summary.csv')
print('补充分析已保存：', dashboard_path)

## 8. 输出与复现实验

引擎自动保存：

- `integrity_audit.csv`：时间、成交和缺失收益审计；
- `run_metadata.json`：因子、路径、参数和时间契约；
- `ic_series.csv` / `ic_summary.csv`：IC 分析；
- `group_returns.csv`：分组、多空、换手及成本；
- `performance.csv`：收益、波动、夏普和回撤；
- `nav_curve.png`：基础净值图；
- `holdings.csv.gz`：仅在 `SAVE_HOLDINGS=True` 时生成。

本 Notebook 额外保存 `yearly_ic.csv`、`group_summary.csv` 和 `research_dashboard.png`。

In [ ]:
metadata_path = OUTPUT_DIR / 'run_metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))

display(pd.Series({
    '因子名称': metadata['factor']['name'],
    '因子列': metadata['factor']['column'],
    '因子方向': metadata['factor']['direction'],
    '因子输入模式': metadata['factor_data']['mode'],
    '因子已预处理': metadata['factor_data']['preprocessed'],
    '信号首日': metadata['actual_sample']['first_signal_date'],
    '信号末日': metadata['actual_sample']['last_signal_date'],
    '回测日数': metadata['actual_sample']['observations'],
    '建仓规则': metadata['time_contract']['entry_rule'],
    '收益规则': metadata['time_contract']['return_rule'],
}).to_frame('值'))

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f'{path.name}: {path.stat().st_size / 1024 / 1024:.2f} MB')

## 9. 接入下一个因子时只检查这些项

1. 修改 `FACTOR_NAME`、`FACTOR_COLUMN`、`FACTOR_DIRECTION` 和 `FACTOR_SOURCE`；
2. 因子若已完成横截面处理，设 `FACTOR_PREPROCESSED=True`；否则设为 `False`；
3. 确认 `backtest.py` 与本 Notebook 位于同一个 `Back_test` 目录；
4. 明确信号在什么时候可知，收盘后因子保持 `EXECUTION_LAG>=1`；
5. `REBALANCE_STEP` 不得小于 `HORIZON`；
6. 训练、验证、测试分开运行，并保持 `ALLOW_EXIT_AFTER_END=False`；
7. 先检查完整性审计，再看 IC 和收益；
8. 参数冻结后才运行验证集和测试集。
